In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from typing import List, Dict
import os

class AdvancedCryptoDataCollector:
    """
    Advanced cryptocurrency data collector with historical OHLCV support
    """
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/stable"
        self.output_dir = "../outputs"
        
    def fetch_crypto_list(self) -> List[Dict]:
        """Fetch complete cryptocurrency list"""
        url = f"{self.base_url}/cryptocurrency-list?apikey={self.api_key}"
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()
            print(f"✓ Retrieved {len(data)} cryptocurrencies")
            return data
        except Exception as e:
            print(f"✗ Error: {e}")
            return []
    
    def fetch_quote(self, symbol: str) -> Dict:
        """Fetch current quote/snapshot data"""
        url = f"{self.base_url}/quote?symbol={symbol}&apikey={self.api_key}"
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()
            return data[0] if data else None
        except:
            return None
    
    def fetch_historical_ohlcv(self, symbol: str, months: int = 24) -> pd.DataFrame:
        """
        Fetch historical OHLCV data for specified period
        Returns DataFrame with: date, open, high, low, close, volume
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=months * 30)
        
        from_date = start_date.strftime('%Y-%m-%d')
        to_date = end_date.strftime('%Y-%m-%d')
        
        url = f"{self.base_url}/historical-price-full/{symbol}?from={from_date}&to={to_date}&apikey={self.api_key}"
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()
            
            if 'historical' in data and data['historical']:
                df = pd.DataFrame(data['historical'])
                df['symbol'] = symbol
                return df
            return pd.DataFrame()
        except:
            return pd.DataFrame()
    
    def calculate_liquidity_metrics(self, hist_df: pd.DataFrame) -> Dict:
        """
        Calculate liquidity and data quality metrics from historical data
        """
        if hist_df.empty:
            return {}
        
        metrics = {}
        
        # Data completeness
        total_days = len(hist_df)
        metrics['data_days_available'] = total_days
        metrics['data_completeness_pct'] = round((total_days / (24 * 30)) * 100, 2)
        
        # Zero volume days
        if 'volume' in hist_df.columns:
            zero_vol_days = (hist_df['volume'] == 0).sum()
            metrics['zero_volume_days'] = zero_vol_days
            metrics['zero_volume_pct'] = round((zero_vol_days / total_days) * 100, 2)
            
            # Average volumes
            metrics['avg_volume_7d'] = hist_df['volume'].tail(7).mean()
            metrics['avg_volume_30d'] = hist_df['volume'].tail(30).mean()
            metrics['avg_volume_90d'] = hist_df['volume'].tail(90).mean()
        
        # Price metrics
        if 'close' in hist_df.columns:
            # Volatility (30, 90, 180 days)
            for period in [30, 90, 180]:
                if len(hist_df) >= period:
                    returns = hist_df['close'].tail(period).pct_change().dropna()
                    metrics[f'volatility_{period}d'] = round(returns.std() * 100, 4)
            
            # Max drawdown
            cumulative = (1 + hist_df['close'].pct_change()).cumprod()
            running_max = cumulative.cummax()
            drawdown = (cumulative - running_max) / running_max
            metrics['max_drawdown_pct'] = round(drawdown.min() * 100, 2)
        
        return metrics
    
    def calculate_statistical_features(self, hist_df: pd.DataFrame, 
                                       btc_hist: pd.DataFrame = None,
                                       eth_hist: pd.DataFrame = None) -> Dict:
        """
        Calculate advanced statistical features
        """
        features = {}
        
        if hist_df.empty:
            return features
        
        # Returns calculation
        if 'close' in hist_df.columns:
            returns = hist_df['close'].pct_change().dropna()
            
            # Sharpe ratio (assuming 0% risk-free rate)
            if len(returns) > 0:
                sharpe = (returns.mean() / returns.std()) * (252 ** 0.5) if returns.std() > 0 else 0
                features['sharpe_ratio'] = round(sharpe, 4)
            
            # Beta vs BTC and ETH
            if btc_hist is not None and 'close' in btc_hist.columns:
                features['beta_vs_btc'] = self._calculate_beta(hist_df, btc_hist)
            
            if eth_hist is not None and 'close' in eth_hist.columns:
                features['beta_vs_eth'] = self._calculate_beta(hist_df, eth_hist)
        
        return features
    
    def _calculate_beta(self, asset_df: pd.DataFrame, market_df: pd.DataFrame) -> float:
        """Calculate beta coefficient"""
        try:
            # Align dates
            merged = asset_df.merge(market_df, on='date', suffixes=('_asset', '_market'))
            
            if len(merged) < 2:
                return None
            
            asset_returns = merged['close_asset'].pct_change().dropna()
            market_returns = merged['close_market'].pct_change().dropna()
            
            if len(asset_returns) < 2 or len(market_returns) < 2:
                return None
            
            covariance = asset_returns.cov(market_returns)
            market_variance = market_returns.var()
            
            beta = covariance / market_variance if market_variance > 0 else None
            return round(beta, 4) if beta is not None else None
        except:
            return None
    
    def collect_comprehensive_data(self, 
                                   top_n: int = None,
                                   include_historical: bool = True,
                                   historical_months: int = 24,
                                   min_market_cap: float = 1e6) -> tuple:
        """
        Collect comprehensive cryptocurrency data
        
        Returns:
            (snapshot_df, historical_df): Two DataFrames
        """
        print("=" * 80)
        print("COMPREHENSIVE CRYPTOCURRENCY DATA COLLECTION")
        print("=" * 80)
        
        # Step 1: Get crypto list
        print("\n[STEP 1/4] Fetching cryptocurrency list...")
        crypto_list = self.fetch_crypto_list()
        
        if not crypto_list:
            return pd.DataFrame(), pd.DataFrame()
        
        # Step 2: Get current snapshot for all
        print(f"\n[STEP 2/4] Fetching current market data...")
        snapshot_data = []
        
        for idx, crypto in enumerate(crypto_list, 1):
            if idx % 100 == 0:
                print(f"  Progress: {idx}/{len(crypto_list)}")
            
            symbol = crypto['symbol']
            quote = self.fetch_quote(symbol)
            
            if quote:
                combined = {**crypto, **quote}
                snapshot_data.append(combined)
            
            time.sleep(0.15)  # Rate limiting
        
        snapshot_df = pd.DataFrame(snapshot_data)
        
        # Filter by market cap
        if 'marketCap' in snapshot_df.columns and min_market_cap > 0:
            initial_count = len(snapshot_df)
            snapshot_df = snapshot_df[snapshot_df['marketCap'] >= min_market_cap]
            print(f"  Filtered to {len(snapshot_df)} cryptos with market cap >= ${min_market_cap:,.0f}")
            print(f"  (Removed {initial_count - len(snapshot_df)} low market cap assets)")
        
        # Sort by market cap and limit
        if 'marketCap' in snapshot_df.columns:
            snapshot_df = snapshot_df.sort_values('marketCap', ascending=False)
            
            if top_n:
                snapshot_df = snapshot_df.head(top_n)
                print(f"  Limited to top {top_n} cryptocurrencies by market cap")
        
        # Step 3: Historical data collection
        historical_df = pd.DataFrame()
        
        if include_historical:
            print(f"\n[STEP 3/4] Fetching historical OHLCV data ({historical_months} months)...")
            print(f"  Processing {len(snapshot_df)} cryptocurrencies...")
            
            all_historical = []
            
            # Get BTC and ETH historical for beta calculation
            print("  Fetching BTC and ETH benchmark data...")
            btc_hist = self.fetch_historical_ohlcv('BTCUSD', historical_months)
            eth_hist = self.fetch_historical_ohlcv('ETHUSD', historical_months)
            time.sleep(0.2)
            
            for idx, row in snapshot_df.iterrows():
                symbol = row['symbol']
                
                if idx % 20 == 0:
                    print(f"    Progress: {idx}/{len(snapshot_df)}")
                
                hist_df = self.fetch_historical_ohlcv(symbol, historical_months)
                
                if not hist_df.empty:
                    all_historical.append(hist_df)
                    
                    # Calculate metrics and add to snapshot
                    liquidity_metrics = self.calculate_liquidity_metrics(hist_df)
                    stat_features = self.calculate_statistical_features(hist_df, btc_hist, eth_hist)
                    
                    # Update snapshot_df with calculated metrics
                    for key, value in {**liquidity_metrics, **stat_features}.items():
                        snapshot_df.loc[snapshot_df['symbol'] == symbol, key] = value
                
                time.sleep(0.2)  # Rate limiting
            
            if all_historical:
                historical_df = pd.concat(all_historical, ignore_index=True)
        else:
            print("\n[STEP 3/4] Skipping historical data collection")
        
        # Step 4: Add final derived features
        print("\n[STEP 4/4] Adding derived features...")
        snapshot_df = self._add_comprehensive_features(snapshot_df)
        
        # Summary
        print("\n" + "=" * 80)
        print("COLLECTION COMPLETE")
        print("=" * 80)
        print(f"Snapshot data: {len(snapshot_df)} cryptocurrencies, {len(snapshot_df.columns)} columns")
        if not historical_df.empty:
            print(f"Historical data: {len(historical_df)} daily records")
            print(f"Average days per crypto: {len(historical_df) / snapshot_df['symbol'].nunique():.0f}")
        print("=" * 80)
        
        return snapshot_df, historical_df
    
    def _add_comprehensive_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add all derived features"""
        
        # Price position metrics
        if all(col in df.columns for col in ['price', 'dayLow', 'dayHigh']):
            df['day_range_pct'] = ((df['dayHigh'] - df['dayLow']) / df['price'] * 100).round(2)
            df['price_position_in_range'] = ((df['price'] - df['dayLow']) / 
                                             (df['dayHigh'] - df['dayLow']) * 100).round(2)
        
        # Year range metrics
        if all(col in df.columns for col in ['price', 'yearLow', 'yearHigh']):
            df['year_range_pct'] = ((df['yearHigh'] - df['yearLow']) / df['price'] * 100).round(2)
            df['price_vs_year_high_pct'] = ((df['price'] - df['yearHigh']) / df['yearHigh'] * 100).round(2)
            df['price_vs_year_low_pct'] = ((df['price'] - df['yearLow']) / df['yearLow'] * 100).round(2)
        
        # Moving average metrics
        if 'price' in df.columns:
            if 'priceAvg50' in df.columns:
                df['vs_50ma_pct'] = ((df['price'] - df['priceAvg50']) / df['priceAvg50'] * 100).round(2)
            if 'priceAvg200' in df.columns:
                df['vs_200ma_pct'] = ((df['price'] - df['priceAvg200']) / df['priceAvg200'] * 100).round(2)
        
        # Volume metrics
        if all(col in df.columns for col in ['volume', 'marketCap']):
            df['volume_mcap_ratio_pct'] = (df['volume'] / df['marketCap'] * 100).round(4)
        
        # Market cap categories
        if 'marketCap' in df.columns:
            df['mcap_category'] = pd.cut(
                df['marketCap'],
                bins=[0, 1e6, 1e7, 1e8, 1e9, 1e10, float('inf')],
                labels=['Nano', 'Micro', 'Small', 'Mid', 'Large', 'Mega']
            )
        
        # Asset age
        if 'icoDate' in df.columns:
            df['asset_age_days'] = (datetime.now() - pd.to_datetime(df['icoDate'], errors='coerce')).dt.days
        
        # Timestamp
        df['data_timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        return df
    
    def save_datasets(self, snapshot_df: pd.DataFrame, historical_df: pd.DataFrame, prefix: str = "crypto"):
        """Save both datasets to CSV"""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Save snapshot
        snapshot_file = f"{prefix}_snapshot_{timestamp}.csv"
        snapshot_path = f"{self.output_dir}/{snapshot_file}"
        snapshot_df.to_csv(snapshot_path, index=False)
        print(f"\n💾 Snapshot saved: {snapshot_file}")
        print(f"   Rows: {len(snapshot_df):,} | Columns: {len(snapshot_df.columns)}")
        
        # Save historical
        if not historical_df.empty:
            historical_file = f"{prefix}_historical_ohlcv_{timestamp}.csv"
            historical_path = f"{self.output_dir}/{historical_file}"
            historical_df.to_csv(historical_path, index=False)
            print(f"💾 Historical saved: {historical_file}")
            print(f"   Rows: {len(historical_df):,} | Columns: {len(historical_df.columns)}")
        
        return snapshot_path, historical_path if not historical_df.empty else None


def main():
    """Main execution"""
    
    API_KEY = "m8TZJWQFGH7G6x2nowAqKdzDfAyakr0T"
    
    collector = AdvancedCryptoDataCollector(API_KEY)
    
    # Configuration
    CONFIG = {
        'top_n': 500,  # Top N by market cap (None = all)
        'include_historical': True,  # Fetch 24-month OHLCV
        'historical_months': 24,
        'min_market_cap': 1e6  # Minimum $1M market cap
    }
    
    print(f"\nConfiguration:")
    print(f"  • Top cryptocurrencies: {CONFIG['top_n']}")
    print(f"  • Historical data: {CONFIG['include_historical']} ({CONFIG['historical_months']} months)")
    print(f"  • Minimum market cap: ${CONFIG['min_market_cap']:,.0f}")
    print()
    
    # Collect data
    snapshot_df, historical_df = collector.collect_comprehensive_data(**CONFIG)
    
    if not snapshot_df.empty:
        # Save datasets
        snapshot_path, historical_path = collector.save_datasets(snapshot_df, historical_df)
        
        # Display preview
        print("\n" + "=" * 80)
        print("SNAPSHOT DATA PREVIEW")
        print("=" * 80)
        print(snapshot_df[['symbol', 'name', 'price', 'marketCap', 'volume', 
                           'changePercentage']].head(10).to_string(index=False))
        
        print("\n" + "=" * 80)
        print("KEY COLUMNS IN SNAPSHOT")
        print("=" * 80)
        for i, col in enumerate(snapshot_df.columns, 1):
            print(f"  {i:2d}. {col}")
        
        if not historical_df.empty:
            print("\n" + "=" * 80)
            print("HISTORICAL DATA PREVIEW")
            print("=" * 80)
            print(historical_df.head(10).to_string(index=False))
        
        # Statistics
        print("\n" + "=" * 80)
        print("DATASET STATISTICS")
        print("=" * 80)
        print(f"Total cryptocurrencies: {len(snapshot_df)}")
        if 'marketCap' in snapshot_df.columns:
            print(f"Total market cap: ${snapshot_df['marketCap'].sum():,.0f}")
            print(f"Average market cap: ${snapshot_df['marketCap'].mean():,.0f}")
            print(f"Median market cap: ${snapshot_df['marketCap'].median():,.0f}")
        if 'volume' in snapshot_df.columns:
            print(f"Total 24h volume: ${snapshot_df['volume'].sum():,.0f}")
        if not historical_df.empty:
            print(f"\nHistorical records: {len(historical_df):,}")
            print(f"Date range: {historical_df['date'].min()} to {historical_df['date'].max()}")
        
        return snapshot_path, historical_path
    
    return None, None


if __name__ == "__main__":
    main()


Configuration:
  • Top cryptocurrencies: 500
  • Historical data: True (24 months)
  • Minimum market cap: $1,000,000

COMPREHENSIVE CRYPTOCURRENCY DATA COLLECTION

[STEP 1/4] Fetching cryptocurrency list...
✓ Retrieved 4785 cryptocurrencies

[STEP 2/4] Fetching current market data...
  Progress: 100/4785
  Progress: 200/4785
  Progress: 300/4785
  Progress: 400/4785
